# 14. 面向对象编程

面向对象编程，简称 OOP，是一种把数据和行为组织到一起的编程思想。

本章重点内容：

- 类和对象
- 属性和方法
- `__init__` 初始化方法
- 实例属性、类属性
- 封装、继承、多态
- 方法重写、`super()`
- 特殊方法，例如 `__str__`
- `@classmethod`、`@staticmethod`
- 组合关系和 `dataclass`

本章不包含练习题，只保留学习笔记和示例代码。


## 1. 类和对象

类是对象的模板，对象是类创建出来的具体实例。

可以把类理解成“图纸”，对象理解成“按照图纸造出来的具体东西”。


In [1]:
class Dog:
    # 类内部定义的函数叫方法
    def bark(self):
        # self 表示当前对象本身
        print('汪汪')


# 创建对象，也叫实例化
dog1 = Dog()
dog2 = Dog()

dog1.bark()
dog2.bark()

print(type(dog1))
print(dog1 is dog2)  # 两次创建得到的是两个不同对象


汪汪
汪汪
<class '__main__.Dog'>
False


### 解释

- `class Dog:` 定义一个类。
- `dog1 = Dog()` 创建一个对象。
- 类中的方法第一个参数通常写 `self`。
- 调用 `dog1.bark()` 时，Python 会自动把 `dog1` 传给 `self`。


## 2. `__init__` 初始化方法

`__init__` 会在创建对象时自动执行，常用于给对象绑定初始属性。


In [2]:
class Student:
    def __init__(self, name, age, score):
        # self.name 是实例属性，每个对象都有自己的属性值
        self.name = name
        self.age = age
        self.score = score

    def introduce(self):
        print(f'我叫{self.name}，今年{self.age}岁，成绩是{self.score}')


student1 = Student('小明', 18, 86)
student2 = Student('小红', 19, 92)

student1.introduce()
student2.introduce()

print(student1.name)
print(student2.name)


我叫小明，今年18岁，成绩是86
我叫小红，今年19岁，成绩是92
小明
小红


### 解释

- `__init__` 不是普通命名，它是 Python 约定好的特殊方法。
- 创建对象时传入的参数会交给 `__init__`。
- `self.name = name` 表示把参数 `name` 保存到当前对象中。
- 不同对象拥有各自独立的实例属性。


## 3. 实例方法和实例属性

实例方法用于操作当前对象的数据。只要方法需要使用对象自己的属性，就应该写成实例方法。


In [3]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        # 存款：修改当前账户的余额
        if amount <= 0:
            print('存款金额必须大于 0')
            return

        self.balance += amount

    def withdraw(self, amount):
        # 取款：需要先判断余额是否足够
        if amount <= 0:
            print('取款金额必须大于 0')
            return

        if amount > self.balance:
            print('余额不足')
            return

        self.balance -= amount

    def show_balance(self):
        print(f'{self.owner} 当前余额：{self.balance}')


account = BankAccount('Tom', 100)
account.deposit(50)
account.withdraw(30)
account.show_balance()


Tom 当前余额：120


### 解释

- `self.balance += amount` 修改的是当前对象的余额。
- 方法内部可以调用或修改对象属性。
- 这种写法把“账户数据”和“账户行为”放在一起，更容易维护。


## 4. 类属性和实例属性

实例属性属于某个对象，类属性属于整个类，所有对象共享。


In [4]:
class Employee:
    # 类属性：所有员工对象共享公司名称
    company = 'OpenAI 学习公司'

    def __init__(self, name, salary):
        # 实例属性：每个员工有自己的姓名和工资
        self.name = name
        self.salary = salary


e1 = Employee('Tom', 8000)
e2 = Employee('Jerry', 9000)

print(e1.company, e1.name, e1.salary)
print(e2.company, e2.name, e2.salary)

# 修改类属性会影响所有还没有同名实例属性的对象
Employee.company = 'Python 学习公司'
print(e1.company)
print(e2.company)

# 给 e1 单独设置 company，会创建一个同名实例属性，只影响 e1
e1.company = '个人工作室'
print(e1.company)
print(e2.company)


OpenAI 学习公司 Tom 8000
OpenAI 学习公司 Jerry 9000
Python 学习公司
Python 学习公司
个人工作室
Python 学习公司


### 解释

- 类属性适合保存所有对象共享的数据。
- 实例属性适合保存每个对象独有的数据。
- 如果实例属性和类属性同名，访问对象属性时会优先找到实例属性。


## 5. 封装和属性控制

封装的核心思想是：对象内部的数据不应该随意被外部直接改坏，可以通过方法或属性控制访问。


In [5]:
class Product:
    def __init__(self, name, price):
        self.name = name
        # 约定：以下划线开头的属性表示内部使用，不建议外部直接访问
        self._price = 0
        self.price = price

    @property
    def price(self):
        # 读取价格时会调用这个方法
        return self._price

    @price.setter
    def price(self, value):
        # 修改价格时会调用这个方法，可以在这里做校验
        if value < 0:
            raise ValueError('价格不能为负数')

        self._price = value


product = Product('键盘', 99)
print(product.price)

product.price = 120
print(product.price)

# product.price = -10
# 上面这行如果取消注释，会抛出 ValueError。


99
120


### 解释

- `_price` 是一种命名约定，表示内部属性。
- `@property` 可以把方法包装成像属性一样访问。
- `@price.setter` 可以控制属性赋值过程。
- 通过这种方式可以防止对象进入非法状态。


## 6. 继承

继承可以让一个类复用另一个类的属性和方法。父类提供通用能力，子类添加或修改自己的特性。


In [6]:
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self):
        print(f'{self.name} 正在吃东西')


class Cat(Animal):
    def meow(self):
        print(f'{self.name} 喵喵叫')


class Dog(Animal):
    def bark(self):
        print(f'{self.name} 汪汪叫')


cat = Cat('小猫')
dog = Dog('小狗')

cat.eat()
cat.meow()

dog.eat()
dog.bark()


小猫 正在吃东西
小猫 喵喵叫
小狗 正在吃东西
小狗 汪汪叫


### 解释

- `class Cat(Animal):` 表示 `Cat` 继承 `Animal`。
- 子类对象可以使用父类的方法。
- 继承适合表达“是一种”的关系，例如猫是一种动物。


## 7. 方法重写和 `super()`

如果子类需要改变父类方法的行为，可以定义同名方法，这叫方法重写。`super()` 可以调用父类方法。


In [7]:
class User:
    def __init__(self, username):
        self.username = username

    def show_info(self):
        print(f'用户：{self.username}')


class VipUser(User):
    def __init__(self, username, level):
        # 调用父类的 __init__，先完成 username 的初始化
        super().__init__(username)
        self.level = level

    def show_info(self):
        # 重写父类方法，展示更多信息
        print(f'VIP 用户：{self.username}，等级：{self.level}')


normal_user = User('Tom')
vip_user = VipUser('Alice', 'Gold')

normal_user.show_info()
vip_user.show_info()


用户：Tom
VIP 用户：Alice，等级：Gold


### 解释

- 子类方法和父类方法同名时，会优先调用子类的方法。
- `super().__init__(username)` 表示复用父类初始化逻辑。
- 重写不是复制粘贴父类代码，而是按需扩展或替换行为。


## 8. 多态

多态表示不同对象可以响应同一个方法名，但具体行为不同。调用者不必关心对象的具体类型，只关心它有没有这个能力。


In [8]:
class Alipay:
    def pay(self, amount):
        print(f'使用支付宝支付 {amount} 元')


class WeChatPay:
    def pay(self, amount):
        print(f'使用微信支付 {amount} 元')


class BankCard:
    def pay(self, amount):
        print(f'使用银行卡支付 {amount} 元')


def checkout(payment, amount):
    # 不关心 payment 是哪个类的对象
    # 只要求它有 pay 方法即可
    payment.pay(amount)


checkout(Alipay(), 100)
checkout(WeChatPay(), 80)
checkout(BankCard(), 60)


使用支付宝支付 100 元
使用微信支付 80 元
使用银行卡支付 60 元


### 解释

- `checkout()` 接收任何拥有 `pay()` 方法的对象。
- 这种风格叫“鸭子类型”：不看对象是什么，只看对象能做什么。
- 多态可以降低代码之间的耦合，让扩展更方便。


## 9. 特殊方法

特殊方法以双下划线开头和结尾，例如 `__str__`、`__len__`。它们可以让自定义对象支持 Python 内置行为。


In [9]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __str__(self):
        # print(book) 时会调用 __str__
        return f'《{self.title}》共 {self.pages} 页'

    def __len__(self):
        # len(book) 时会调用 __len__
        return self.pages


book = Book('Python 入门', 320)

print(book)
print(len(book))


《Python 入门》共 320 页
320


### 解释

- `__str__` 用于定义对象转换成字符串时的展示效果。
- `__len__` 用于支持 `len()`。
- 特殊方法不是随便调用的，通常由 Python 的内置函数或语法自动触发。


## 10. 类方法和静态方法

实例方法操作对象，类方法操作类，静态方法只是放在类里的普通工具函数。


In [10]:
class DateTool:
    date_format = 'YYYY-MM-DD'

    @classmethod
    def show_format(cls):
        # cls 表示当前类本身
        print('日期格式：', cls.date_format)

    @staticmethod
    def is_leap_year(year):
        # 静态方法不需要 self，也不需要 cls
        return year % 4 == 0 and year % 100 != 0 or year % 400 == 0


DateTool.show_format()
print(DateTool.is_leap_year(2024))
print(DateTool.is_leap_year(2025))


日期格式： YYYY-MM-DD
True
False


### 解释

- `@classmethod` 的第一个参数通常写 `cls`，表示类。
- `@staticmethod` 没有自动传入的 `self` 或 `cls`。
- 如果方法需要访问实例属性，用实例方法。
- 如果方法需要访问或修改类属性，用类方法。
- 如果只是逻辑上属于这个类的工具函数，用静态方法。


## 11. 组合和 dataclass

组合表示一个对象拥有另一个对象。相比继承，组合更适合表达“有一个”的关系。


In [11]:
from dataclasses import dataclass


@dataclass
class Address:
    city: str
    street: str


@dataclass
class Customer:
    name: str
    address: Address

    def show_info(self):
        print(f'{self.name} 住在 {self.address.city} {self.address.street}')


address = Address(city='上海', street='人民路')
customer = Customer(name='Tom', address=address)

customer.show_info()
print(customer)


Tom 住在 上海 人民路
Customer(name='Tom', address=Address(city='上海', street='人民路'))


### 解释

- `Customer` 拥有一个 `Address`，这就是组合关系。
- `@dataclass` 可以自动生成 `__init__`、`__repr__` 等方法。
- 数据类适合表示以保存数据为主的简单对象。


## 12. 常见错误总结

1. 定义方法时忘记写 `self`。
2. 调用实例方法时把类当对象用。
3. 在类属性中保存会被修改的列表或字典，导致所有对象共享同一份数据。
4. 继承层级过深，导致代码难理解。
5. 为了使用类而使用类，把简单函数强行包装成类。
6. 误以为 `_name` 是绝对私有；它只是约定，不是强制限制。
7. 子类重写 `__init__` 时忘记调用 `super().__init__()`。
8. 把“有一个”关系写成继承，导致设计不自然。
9. 特殊方法拼写错误，例如把 `__init__` 写成 `_init_`。
10. 一个类承担太多职责，后期维护困难。
